# Time Operators

Streaming data is infinite. Kafi Streams is an in-memory stream processor. Memory is never infinite.

So of course you need *time operators* that effectively clean up memory so that your Kafi Streams processing pipeline has constant, not ever-growing memory usage.

Freeing memory of timed out data is implemented as [expiry](#expiry) in Kafi Streams.

The time operators also allow you to implement the [*time windows*](#windows) as e.g. in Kafka Streams, such as [tumbling](#tumbling), [hopping](#hopping), [cumulative](#cumulative), [sliding](#sliding) and [session](#session) windows. And moreover, Kafi Streams enables you to build arbitrary [new types of time windows](#custom) as well.

## Overview

[Preparation](#prep)

* [Expiry](#expiry)
  * [expire()](#expire-operator)
* [Time windows](#windows)
  * [Time Windows = expire + group + aggregate](#expire_group_aggregate)
  * [Tumbling windows](#tumbling)
  * [Hopping windows](#hopping)
  * [Cumulative windows](#cumulative)
  * [Sliding windows](#sliding)
  * [Session windows](#session)
  * [Triggers and Custom windows](#custom)
  

---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [75]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator, OrderGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()
order_generator = OrderGenerator()

click_source_str = "clicks"
customer_source_str = "customers"
order_source_str = "orders"
sink_str = "sink"

#

def run(built_tn):
    sink_m_list = []
    for i in range(100):
        # 1. Generate new data.
        click_m_list = click_generator.generate(100)
        customer_m_list = customer_generator.generate(100)

        # 2. Push the new data to the topology + incrementally process the new data + get the resulting changes.
        sink_str_m_list_dict = built_tn.process({click_source_str: click_m_list, customer_source_str: customer_m_list})
        m_list = sink_str_m_list_dict[sink_str]

        # 3. Print out the size of the pydbsp state.
        sys.stdout.write(f"\rStep: {i + 1}, Memory: {built_tn.get_state_size() / 1024}KB")

        # 4. Add the changes to the output list.
        sink_m_list += m_list

    print()
    print(len(sink_m_list))
    print(sink_m_list[-10:])

def process(built_tn, customer_id, price, ts, w=1):
    m = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    #
    sink_str_r_w_tuple_list_dict = built_tn.process({order_source_str: [(m, w)]})
    r_w_tuple_list = sink_str_r_w_tuple_list_dict[sink_str]
    r_w_tuple_list = [(r, w) for r, w in r_w_tuple_list if w != 0]
    #
    return r_w_tuple_list

def assert_output(actual_r_w_tuple_list, expected_r_w_tuple_list):
    def sort_key(r_w_tuple):
        r, w = r_w_tuple
        return (r.get("window_end", 0), r.get("customer_id", 0), w)
    #
    actual_sorted_r_w_tuple_list = sorted(actual_r_w_tuple_list, key=sort_key)
    expected_sorted_r_w_tuple_list = sorted(expected_r_w_tuple_list, key=sort_key)
    #
    if actual_sorted_r_w_tuple_list != expected_sorted_r_w_tuple_list:
        raise ValueError(f"\nAssertion Failed!\nExpected: {expected_sorted_r_w_tuple_list}\nGot:      {actual_sorted_r_w_tuple_list}")




[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Please also note that when we re-use the same example over and over again to illustrate how the operators work, we always mark the important new parts as follows:
```python
    # <------------------------------>
    ...important new parts...
    # <------------------------------>
```

---
<a id="expiry"></a>
## Expiry

Expiry is the central concept in Kafi Streams for freeing memory of timed out data.

Thanks to pydbsp, Kafi Streams can implement expiry natively, without having to bolt on any kind of mechanism on top.

Essentially, expiry has to be defined only once for each source at the beginning of the Kafi Streams topology. All the stateful operators downstream do not need any special handling - they are automatically cleaned up by the expired records percolating through the topology, one by one.

We need an example. Let us recollect the example from the [Quickstart](../quickstart.ipynb) using the `TopologyNode` class (see [Architecture](../architecture.ipynb))


In [ ]:
click_source_str = "clicks"
customer_source_str = "customers"

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

sink_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
    .sink(sink_str)
)

built_tn = Tn.build(sink_tn)


And then, let us throw data at it and see how the global state size of the topology grows:

In [ ]:
built_tn.reset()
for _ in range(3):
    run(built_tn)

This is of course not sustainable. The memory usage grows unboundedly and Kafi Streams gets slower and slower (in this case, mostly caused by the join).

It's time to introduce the `expire()` operator.

<a id="expire-operator"></a>
### expire()

Expires individual records so that they can be purged from memory.

```
expire(ts_fun, expiry_fun, project_fun=lambda r_ts_tuple: r_ts_tuple[0], **kwargs)
```
* `ts_fun: r -> ts`: timestamp function - gets an input record and returns a timestamp
* `expiry_fun: ts -> ts`: expiry function - gets a timestamp and returns the corresponding expiry timestamp specifying when the record shall expire
* `project_fun: tuple(r, ts) -> r`: projection function - gets a pair of a record and its expiry timestamp and returns another record. Default: `lambda r_ts_tuple: r_ts_tuple[0]`

Let's add this to our topology.


In [ ]:
click_source_str = "clicks"
customer_source_str = "customers"

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    # <------------------------------>
    .expire(ts_fun=lambda r: r["ts"],
            expiry_fun=lambda ts: ts + click_generator.ts_step_int * 1000)
    # <------------------------------>
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

sink_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
    .sink(sink_str)
)

built_tn = Tn.build(sink_tn)


Here, we use the `expiry()` operator to:
1. Select the `ts` field from each record,
2. and then set the expiry to the selected timestamp plus `1000` times the timestamp step size of the click generator.

When you look closer at the topology, you can observe multiple some of the properties of expiry in Kafi Streams:
* It is defined once at the top of the topology, before any stateful operator (`map` and `filter` are stateless).
* The stateful operators (`distinct` and `join_equi`) below in the topology do not need to know anything about the expiry.

Ok. Let's see this in action. Will be able to rein in the memory consumption?

In [ ]:
for _ in range(5):
    run(built_tn)

It works! Constant, flat memory usage! No processing slowdown!

Why? Because we used `expire()` to time out the transactional data (=the clicks). After a short while, the memory consumption of the master data (=the customers) becomes constant as well because it is limited (the generator only generates up to 100 customers in the example).

---
<a id="windows"></a>
## Time windows

In the previous section, we learnt how we can keep Kafi Streams' memory usage at check. It was only remotely related to time windows in the classical stream processing sense: under the covers, the `expire()` operator works akin to a "sliding window" in classical stream processing.

This section is about "real" stream processing time windows.

You'll see that we devised a novel formulation of them inside DBSP that allows us to build all the time window types from classical stream processing.

But it doesn't stop there - Kafi Streams is so flexible that you can easily build your own custom time windows.


<a id="expire_group_aggregate"></a>
### Time Windows = expire + group + aggregate

What is a time window really? You can think of time windows in stream processing as consisting of two ingredients:
* **Expiry**: Time windows have a start and an end. Events *expire* after the end of a time window so that they can be cleaned up.
* **Group By + Aggregate**: The actual "time window" is a set of events *grouped* by time and some other key (e.g. a customer ID) and *aggregated*.

Now this is very theoretical. Let's pick the simplest time window - the *tumbling window* and see how all this theory plays out in practice.

In [ ]:
# <------------------------------>

def ts_fun(r):
    return r["ts"]

size_int = click_generator.ts_step_int * 1000

# <------------------------------>

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    
    # <------------------------------>
    
    .expire_tumbling(ts_fun=ts_fun,
                     size_int=size_int)
    
    # <------------------------------>
    
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

joined_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]})
)

# <------------------------------>

sink_tn = (
    joined_tn
    .group_by_agg_tumbling(
        ts_fun=ts_fun,
        size_int=size_int,
        key_fun=lambda r: {"customer_id": r["customer_id"], "name": r["name"]},
        agg_fun=lambda agg_r, r: {"clicks": agg_r["clicks"] + 1,
                                  "view_times": agg_r["view_times"] + [r["view_time"]],
                                  "total_view_time": agg_r["total_view_time"] + r["view_time"]},
        agg_initial_any={"clicks": 0, "view_times": [], "total_view_time": 0},
        project_fun=lambda key_any, agg_r: {"customer_id": key_any["customer_id"],
                                            "name": key_any["name"],
                                            "clicks": agg_r["clicks"],
                                            "view_times": agg_r["view_times"],
                                            "total_view_time": agg_r["total_view_time"]})
    .sink(sink_str)
)

# <------------------------------>

built_tn = Tn.build(sink_tn)


What do we do here?

* **expire**: At the top of the topology, we specify the expiry of the incoming clicks using the `expire_tumbling()` operator and set the tumbling window size to `tumbling_size_int = click_generator.ts_step_int * 1000`.
* **group + aggregate**: After the join of clicks and customers, we create the tumbling window using the `group_by_agg_tumbling()` operator: We group by `customer_id` and `name`, and aggregate the clicks for that customer in that time window:
  * `clicks` the number of clicks of the customer
  * `view_times` the list of view times of the customer
  * `total_view_time` the total view time of the customer

Let's run this.

In [ ]:
run(built_tn)

This was your first time window in Kafi Streams in action!

In the following sections, we double down on the individual built-in window types of Kafi Streams and explain their API in detail.

<a id="tumbling"></a>
### Tumbling Windows

In Kafi Streams, the two operators required to set up a tumbling window are `expire_tumbling` and `group_by_agg_tumbling`.

<a id="expire_tumbling-operator"></a>
#### expire_tumbling()

Syntactic sugar for `expire()` for record expiry in the context of tumbling windows.

```
expire_tumbling(ts_fun, size_int, allowed_lateness_int=0, **kwargs)
```
* `ts_fun: r -> ts` timestamp function - gets an input record and returns a timestamp
* `size_int` the size (in milliseconds) of the tumbling window
* `allowed_lateness_int` allowed lateness - adds a time (in milliseconds) to the expiry time to accommodate late arriving records


<a id="group_by_agg_tumbling-operator"></a>
#### group_by_agg_tumbling()

Augmented `group_by_agg()` operator for creating time windows by grouping by + aggregating. Implicitly also groups by time windows and triggers the emission of aggregated time windows.

```
group_by_agg_tumbling(ts_fun, size_int, key_fun, agg_fun, agg_initial_any, project_fun, trigger_fun=lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1], trigger_project_fun=lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]}, trigger_positive_only_bool=True, **kwargs)
```

* `ts_fun: r -> ts`: timestamp function - gets an input record and returns a timestamp
* `size_int`: the size (in milliseconds) of the tumbling window
* `key_fun: r -> any`: the selection function for the key for the grouping
* `agg_fun: agg_any, value_any -> any`: the aggregation function; gets the result of the aggregation so far (`agg_any`) and the next selected value to aggregate (`value_any`), and returns the updated aggregation
* `agg_initial_any`: the initial value of the aggregation
* `project_fun: key_any, agg_any -> r`: the projection function; gets both the selected key and the aggregation result for that key and returns the output (=projection) of the group by + aggregation
* `trigger_fun: tuple(r, end_ts), latest_ts -> bool` the trigger function; gets a pair of a record and the window end timestamp and the latest timestamp and returns a bool. `True` for triggering the emission of the output, `False` for not yet triggering it.
* `trigger_project_fun: tuple(r, end_ts) -> r`: the trigger projection function; gets a pair of a record and the window end timestamp and returns a record. Default: `lambda r_end_ts_tuple: {**r_end_ts_int_tuple[0], "window_end": r_end_ts_int_tuple[1]}`, i.e., add the window end timestamp to the record.
* `trigger_positive_only_bool`: trigger only updates with positive weights or also zero or negative ones. Default: `True`

This looks scary at first. But for most use cases, only these parameters are obligatory:
* `ts_fun` - same as in `expire_tumbling()`
* `size_int` - same as in `expire_tumbling()`
* `key_fun` - as in `group_by_agg()`
* `agg_fun` - as in `group_by_agg()`
* `agg_initial_any` - as in `group_by_agg()`
* `project_fun` - as in `group_by_agg()`

The `trigger_` parameters should only be required for advanced use cases. They control the emission of time windows based on a triggering mechanism. Their defaults should suffice for most uses cases. We'll show a use case for them when we discuss custom time windows at the end of this notebook.

Note that contrary to the basic `group_by_agg()` operator, there is no `value_fun`. This is because the `value_fun` in `group_by_agg_tumbling` is always the identity function since we assume that in 99% of the use cases, you would want to create time windows around entire records.

Ok. After so many parameters, we need to see them in action. Here is an example, this time not about clicks and customers, but just orders to keep it simple:


In [ ]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

size_int = order_generator.ts_step_int * 100
allowed_lateness_int = size_int * 2
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    # 1. Select customer_id, price and ts from the value.
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    # 2. Expire with window size order_generator.ts_step_int * 100 and allowed_lateness = window_size * 2,  
    .expire_tumbling(lambda r: r["ts"], size_int, allowed_lateness_int)
    # 3. Deduplicate.
    .distinct()
)
#
# 4. Set up the tumbling window: group by customer ID, count the orders and sum up the prices of the orders.
sink_tn = order_tn.group_by_agg_tumbling(
    ts_fun=lambda r: r["ts"],
    size_int=size_int,
    key_fun=lambda r: r["customer_id"],
    agg_fun=lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                              "total_price": agg_r["total_price"] + r["price"]},
    agg_initial_any={"orders": 0, "total_price": 0},
    project_fun=lambda key_any, agg_r: {"customer_id": key_any,
                                        "orders": agg_r["orders"],
                                        "total_price": agg_r["total_price"]},
    trigger_positive_only_bool=False
).sink(sink_str)
#
built_tn = Tn.build(sink_tn)
_ = built_tn.from_zSet(Tn._to_records)

What do we do?
1. `map()`: Select customer_id, price and ts from the value.
2. `expire_tumbling()`: Expire with window size order_generator.ts_step_int * 100 and allowed_lateness = window_size * 2,  
3. `distinct()`: Deduplicate.
4. `group_by_agg_tumbling`: Set up the tumbling window: group by customer ID, count the orders and sum up the prices of the orders.

Next, we illustrate how the tumbling window works by processing some example data - one by one, in baby steps:


#### Step 1 - Two orders from customer 1 arrive (at timestamps 10, 50)

```mermaid
flowchart LR
    subgraph x_axis ["Current maximum timestamp = 50"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> space1[ ]
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style space1 fill:none,stroke:none
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#1565c0,stroke-width:6px
    E10[ ]
    style E10 fill:none,stroke:none
    10 ~~~ E10

    50@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}"}
    style 50 fill:none,stroke:#1565c0,stroke-width:6px
    E50[ ]
    style E50 fill:none,stroke:none
    50 ~~~ E50
```


#### Step 2 - An order from customer 2 arrives at timestamp 105

```mermaid
flowchart LR
    subgraph x_axis ["Current maximum timestamp = 105"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 105 e5@-.-> 200(("200")) e6@-.-> space1[ ]
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style space1 fill:none,stroke:none
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px
    E10[ ]
    style E10 fill:none,stroke:none
    10 ~~~ E10

    50@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}"}
    style 50 fill:none,stroke:#333,stroke-width:6px
    E50[ ]
    style E50 fill:none,stroke:none
    50 ~~~ E50

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#1565c0,stroke-width:6px
    E105[ ]
    style E105 fill:none,stroke:none
    105 ~~~ E105

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 2,\n&quot;total_price&quot;: 300}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    105 Link@== Triggers ==> Output
    linkStyle 9 stroke:#00bb00,stroke-width:3px;
```

#### Step 3 - The order from customer 1 is retracted (at timestamp 50)

```mermaid
flowchart LR
    subgraph x_axis ["Current maximum timestamp = 105"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 105 e5@-.-> 200(("200")) e6@-.-> space1[ ]
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style space1 fill:none,stroke:none
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px
    E10[ ]
    style E10 fill:none,stroke:none
    10 ~~~ E10

    50@{ shape: lean-r, label: "<s style='color:red;'>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#bb0000,stroke-width:6px
    E50[ ]
    style E50 fill:none,stroke:none
    50 ~~~ E50

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px
    E105[ ]
    style E105 fill:none,stroke:none
    105 ~~~ E105

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 100}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    50 Link@== Triggers ==> Output
    linkStyle 9 stroke:#00bb00,stroke-width:3px;
```

#### Step 4 - An order from customer 3 arrives (at timestamp 101)

```mermaid
flowchart LR
    subgraph x_axis ["Current maximum timestamp = 105"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 200(("200")) e7@-.-> space1[ ]
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style space1 fill:none,stroke:none
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px
    E10[ ]
    style E10 fill:none,stroke:none
    10 ~~~ E10

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px
    E50[ ]
    style E50 fill:none,stroke:none
    50 ~~~ E50

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px
    E105[ ]
    style E105 fill:none,stroke:none
    105 ~~~ E105

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#1565c0,stroke-width:6px
    E101[ ]
    style E101 fill:none,stroke:none
    101 ~~~ E101
```

#### Step 5 - Another order from customer 3 arrives (at timestamp 150)

```mermaid
flowchart LR
    subgraph x_axis ["Current maximum timestamp = 150"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100 e4@-.-> 105 e5@-.-> 150 e6@-.-> 200(("200")) e7@-.-> space1[ ]
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style space1 fill:none,stroke:none
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px
    E10[ ]
    style E10 fill:none,stroke:none
    10 ~~~ E10

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px
    E50[ ]
    style E50 fill:none,stroke:none
    50 ~~~ E50

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px
    E105[ ]
    style E105 fill:none,stroke:none
    105 ~~~ E105

    100@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 100}"}
    style 100 fill:none,stroke:#333,stroke-width:6px
    E100[ ]
    style E100 fill:none,stroke:none
    100 ~~~ E100

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#1565c0,stroke-width:6px
    E150[ ]
    style E150 fill:none,stroke:none
    150 ~~~ E150
```

In [134]:
built_tn.reset()

print("=== Step 1: Two orders from customer 1 (price=100, ts=10) and (price=200, ts=50) arrive ===")
process(built_tn, customer_id=1, price=100, ts=10, w=1)
process(built_tn, customer_id=1, price=200, ts=50, w=1)
print("-> OK")

print("\n=== Step 2: An order from customer 2 (price=50, ts=105) arrives ===")
r_w_tuple_list = process(built_tn, customer_id=2, price=50, ts=105, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, 1)
])
print("-> OK: Window [0, 100) triggered (orders=2, total=300).")

print("\n=== Step 3: The order from customer 1 at 50 is retracted ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=200, ts=50, w=-1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, -1),
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, 1)
])
print("-> OK: Correction for window [0, 100) triggered: (customer=1, orders=1, total=100).")

print("\n=== Step 4: The order from customer 1 at 10 is also retracted ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=100, ts=10, w=-1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, -1)
])
print("-> OK: Retraction for window [0, 100) triggered.")

print("\n=== Step 5: An order from customer 3 (price=400, ts=100) arrives ===")
r_w_tuple_list = process(built_tn, customer_id=3, price=400, ts=100, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order put into window [100, 200), still correctly held back/not triggered.")

print("\n=== Step 6: Another order from customer 3 at 150 (price=200, ts=150) arrives ===")
r_w_tuple_list = process(built_tn, customer_id=3, price=200, ts=150, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK. Order put into window [100, 200), still correctly held back/not triggered.")

print("\n=== Step 7: Another order from customer 1 (price=50, ts=210) arrives ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=50, ts=210, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 2, "total_price": 600, "window_end": 200}, 1)
])
print("-> OK: Window [100, 200) triggered (customer=2, orders=1, total=50), (customer=3, orders=2, total=600).")

print("\n=== Step 8: An order from customer 2 (price=60, ts=310) arrives ===")
r_w_tuple_list = process(built_tn, customer_id=2, price=60, ts=310, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 50, "window_end": 300}, 1)
])
print("-> OK: Window [200, 300) triggered (customer=1, orders=1, total=50).")

print("\n=== Step 9: Order from customer 2 (price=40, ts=120) arrives late (but not too late) ===")
r_w_tuple_list = process(built_tn, customer_id=2, price=40, ts=120, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 200}, -1),
    ({"customer_id": 2, "orders": 2, "total_price": 90, "window_end": 200}, 1)
])
print("-> OK: Correction for window [100, 200) triggered: (customer=2, orders=2, total=90).")

print("\n=== Step 10: Order from customer 1 (price=70, ts=510) arrives ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=70, ts=510, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 2, "total_price": 90, "window_end": 200}, -1),
    ({"customer_id": 3, "orders": 2, "total_price": 600, "window_end": 200}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 60, "window_end": 400}, 1)
])
print("-> OK: Retraction for window [100, 200) triggered. Window (300, 400) triggered: (customer=2, orders=1, total=60).")

print("\n=== Step 11: Order from customer 1 (price=70, ts=130) arrives too late ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=70, ts=130, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")

print("\n🎉 Done.")

=== Step 1: Two orders from customer 1 (price=100, ts=10) and (price=200, ts=50) arrive ===
-> OK

=== Step 2: An order from customer 2 (price=50, ts=105) arrives ===
-> OK: Window [0, 100) triggered (orders=2, total=300).

=== Step 3: The order from customer 1 at 50 is retracted ===
-> OK: Correction for window [0, 100) triggered: (customer=1, orders=1, total=100).

=== Step 4: The order from customer 1 at 10 is also retracted ===
-> OK: Retraction for window [0, 100) triggered.

=== Step 5: An order from customer 3 (price=400, ts=100) arrives ===
-> OK: Order put into window [100, 200), still correctly held back/not triggered.

=== Step 6: Another order from customer 3 at 150 (price=200, ts=150) arrives ===
-> OK. Order put into window [100, 200), still correctly held back/not triggered.

=== Step 7: Another order from customer 1 (price=50, ts=210) arrives ===
-> OK: Window [100, 200) triggered (customer=2, orders=1, total=50), (customer=3, orders=2, total=600).

=== Step 8: An order

In [129]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

size_int = order_generator.ts_step_int * 100
hop_int = size_int // 2
allowed_lateness_int = size_int * 2
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_hopping(lambda r: r["ts"], size_int, hop_int, allowed_lateness_int)
    .distinct()
)
#
sink_tn = order_tn.group_by_agg_hopping(
    ts_fun=lambda r: r["ts"],
    size_int=size_int,
    hop_int=hop_int,
    key_fun=lambda r: r["customer_id"],
    agg_fun=lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                              "total_price": agg_r["total_price"] + r["price"]},
    agg_initial_any={"orders": 0, "total_price": 0},
    project_fun=lambda key_any, agg_r: {"customer_id": key_any,
                                        "orders": agg_r["orders"],
                                        "total_price": agg_r["total_price"]},
    trigger_positive_only_bool=False
).sink(sink_str)                                       
#
built_tn = Tn.build(sink_tn)
built_tn.from_zSet(Tn._to_records)

In [130]:
built_tn.reset()

print("=== Step 1: Two orders from customer 1 arrive (ts=10 falls into [0, 100); ts=60 falls into [0, 100) and [50, 150)) ===")
process(built_tn, customer_id=1, price=100, ts=10, w=1)
process(built_tn, customer_id=1, price=200, ts=60, w=1)
print("-> OK.")

print("\n=== Step 2: An order from customer 2 at ts=110 arrives ===")
r_w_tuple_list = process(built_tn, customer_id=2, price=50, ts=110, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, 1)
])
print("-> OK: Window [0, 100) triggered: (customer=1, orders=2, total=300)")

print("\n=== Step 3: Retraction for the window (50, 150) arrives ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=200, ts=60, w=-1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, -1),
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, 1)
])
print("-> OK: Correction for window [0, 100) triggered: (customer=1, orders=1, total=100).")

print("\n=== Step 4: An order from customer 3 at arrives (price=99, ts=150) ===")
r_w_tuple_list = process(built_tn, customer_id=3, price=99, ts=150, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 150}, 1)
])
print("-> OK: Window [50, 150) triggered: (customer=2, orders=2, total=50).")

print("\n=== Step 5: An order from customer 2 arrives (price=99, ts=250) ===")
r_w_tuple_list = process(built_tn, customer_id=2, price=90, ts=250, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 1, "total_price": 99, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 1, "total_price": 99, "window_end": 250}, 1)
])
print("-> OK: Windows [100, 200) and [150, 250) triggered.")

print("\n=== Step 6: An order for customer 3 (price=100, ts=170) arrives late (but not too late) ===")
r_w_tuple_list = process(built_tn, customer_id=3, price=100, ts=170, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 3, "orders": 1, "total_price": 99, "window_end": 200}, -1),
    ({"customer_id": 3, "orders": 2, "total_price": 199, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 1, "total_price": 99, "window_end": 250}, -1),
    ({"customer_id": 3, "orders": 2, "total_price": 199, "window_end": 250}, 1)
])
print("-> OK: Corrections for windows [100, 200) and [150, 250) triggered.")

print("\n=== Step 7: An order for customer 1 (price=10, ts=450) arrives ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=10, ts=450, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 90, "window_end": 300}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 90, "window_end": 350}, 1)
])
print("-> OK: Window [0, 100) correctly expired; windows [200, 300) and [250, 300) correctly triggered.")

print("\n=== Step 8: An order from customer 1 (price=999, ts=20) arrives too late ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=999, ts=20, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")

print("\n🎉 Done.")


=== Step 1: Two orders from customer 1 arrive (ts=10 falls into [0, 100); ts=60 falls into [0, 100) and [50, 150)) ===
-> OK.

=== Step 2: An order from customer 2 at ts=110 arrives ===
-> OK: Window [0, 100) triggered: (customer=1, orders=2, total=300)

=== Step 3: Retraction for the window (50, 150) arrives ===
-> OK: Correction for window [0, 100) triggered: (customer=1, orders=1, total=100).

=== Step 4: An order from customer 3 at arrives (price=99, ts=150) ===
-> OK: Window [50, 150) triggered: (customer=2, orders=2, total=50).

=== Step 5: An order from customer 2 arrives (price=99, ts=250) ===
-> OK: Windows [100, 200) and [150, 250) triggered.

=== Step 6: An order for customer 3 (price=100, ts=170) arrives late (but not too late) ===
-> OK: Corrections for windows [100, 200) and [150, 250) triggered.

=== Step 7: An order for customer 1 (price=10, ts=450) arrives ===
-> OK: Window [0, 100) correctly expired; windows [200, 300) and [250, 300) correctly triggered.

=== Step 8: 

In [127]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

size_int = order_generator.ts_step_int * 100
advance_int = size_int // 5
allowed_lateness_int = size_int * 2
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_cumulative(lambda r: r["ts"], size_int, advance_int, allowed_lateness_int)
    .distinct()
)
#
sink_tn = order_tn.group_by_agg_cumulative(
    ts_fun=lambda r: r["ts"],
    size_int=size_int,
    advance_int=advance_int,
    key_fun=lambda r: r["customer_id"],
    agg_fun=lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                              "total_price": agg_r["total_price"] + r["price"]},
    agg_initial_any={"orders": 0, "total_price": 0},
    project_fun=lambda key_any, agg_r: {"customer_id": key_any,
                                        "orders": agg_r["orders"],
                                        "total_price": agg_r["total_price"]},
                                        trigger_positive_only_bool=False
).sink(sink_str)
#
built_tn = Tn.build(sink_tn)
built_tn.from_zSet(Tn._to_records)


In [128]:
built_tn.reset()

print("=== Step 1: An order for customer 1 arrives (price=100, ts=10). Lands in [0, 20), [0, 40), [0, 60), [0, 80), [0, 100) ===")
# 
process(built_tn, customer_id=1, price=100, ts=10, w=1)
print("-> OK.")


print("=== Step 2: Another order for customer 1 arrives (price=200, ts=30). Lands in [0, 40), [0, 60), [0, 80), [0, 100) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=200, ts=30, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 20}, 1)
])
print("-> OK: Window [0, 20) triggered.")


print("\n=== Step 3: An order for customer 2 arrives (price=50, ts=75). Lands in [60, 80), [80, 100) ===")
r_w_tuple_list = process(built_tn, customer_id=2, price=50, ts=75, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 40}, 1),
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 60}, 1)
])
print("-> OK: Windows [0, 40) and [0, 60) triggered.")


print("\n=== Step 4: Another order from customer 1 (price=50, ts=15) arrives late (but not too late) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=50, ts=15, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 20}, -1),
    ({"customer_id": 1, "orders": 2, "total_price": 150, "window_end": 20}, 1),
    
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 40}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 40}, 1),
    
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 60}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 60}, 1)
])
print("-> OK: Corrections for windows [0, 20), [0, 40) and [0, 60) triggered.")


print("\n=== Step 5: Another order from customer 1 (price=500, ts=105) arrives ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=500, ts=105, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 80}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 80}, 1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 100}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 100}, 1)
])
print("-> OK: Windows [0, 80) and [0, 100] triggered.")


print("\n=== Step 6: Yet another order from customer 1 (price=10, ts=410) arrives ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=10, ts=410, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 150, "window_end": 20}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 40}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 60}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 80}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 80}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 100}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 100}, -1),
    #    
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 120}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 140}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 160}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 180}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 200}, 1)
])
print("-> OK: All windows until 200) triggered.")


print("\n=== Step 7: Another order from customer 1 (price=999, ts=10) arrives too late ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=999, ts=10, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")


print("\n🎉 Done.")


=== Step 1: An order for customer 1 arrives (price=100, ts=10). Lands in [0, 20), [0, 40), [0, 60), [0, 80), [0, 100) ===
-> OK.
=== Step 2: Another order for customer 1 arrives (price=200, ts=30). Lands in [0, 40), [0, 60), [0, 80), [0, 100) ===
-> OK: Window [0, 20) triggered.

=== Step 3: An order for customer 2 arrives (price=50, ts=75). Lands in [60, 80), [80, 100) ===
-> OK: Windows [0, 40) and [0, 60) triggered.

=== Step 4: Another order from customer 1 (price=50, ts=15) arrives late (but not too late) ===
-> OK: Corrections for windows [0, 20), [0, 40) and [0, 60) triggered.

=== Step 5: Another order from customer 1 (price=500, ts=105) arrives ===
-> OK: Windows [0, 80) and [0, 100] triggered.

=== Step 6: Yet another order from customer 1 (price=10, ts=410) arrives ===
-> OK: All windows until 200) triggered.

=== Step 7: Another order from customer 1 (price=999, ts=10) arrives too late ===
-> OK: Order arrived too late - no action.

🎉 Done.


In [125]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

size_int = order_generator.ts_step_int * 100
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_sliding(lambda r: r["ts"], size_int)
)
#
sink_tn = order_tn.group_by_agg_sliding(
    ts_fun=lambda r: r["ts"],
    size_int=size_int,
    key_fun=lambda r: r["customer_id"],
    agg_fun=lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                              "total_price": agg_r["total_price"] + r["price"]},
    agg_initial_any={"orders": 0, "total_price": 0},
    project_fun=lambda key_any, agg_r: {"customer_id": key_any,
                                        "orders": agg_r["orders"],
                                        "total_price": agg_r["total_price"]},
    trigger_positive_only_bool=False
).sink(sink_str)
#
built_tn = Tn.build(sink_tn)
built_tn.from_zSet(Tn._to_records)


In [126]:
built_tn.reset()

print("=== Step 1: An order from customer 1 arrives (price=100, ts=10) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=100, ts=10, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 110}, 1)
])
print("-> OK. Window [10, 110) triggered correctly.")


print("\n=== Step 2: Another order from customer 1 arrives (price=200, ts=30) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=200, ts=30, w=1)
assert_output(r_w_tuple_list, [
    ({'customer_id': 1, 'orders': 1, 'total_price': 100, 'window_end': 110}, -1),
    ({'customer_id': 1, 'orders': 2, 'total_price': 300, 'window_end': 110}, 1)]
)
print("-> OK: Correction for window [10, 110) triggered correctly.")


print("\n=== Step 3: An order from customer 2 arrives (price=50, ts=75) ===")
r_w_tuple_list = process(built_tn, customer_id=2, price=50, ts=75, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 175}, 1)
])
print("-> OK: Window [75, 175) triggered correctly.")


print("\n=== Step 4: Another order from customer 1 arrives late but still inside [10, 110) (price=50, ts=15) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=50, ts=15, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 110}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 110}, 1)
])
print("-> OK: Window [10, 110) retracted correctly; window [15, 11ß) triggered correctly.")


print("\n=== Step 5: Yet another order from customer 1 arrives (not inside [15, 115) (price=500, ts=200) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=500, ts=200, w=1)
assert_output(r_w_tuple_list, [
    ({'customer_id': 1, 'orders': 3, 'total_price': 350, 'window_end': 110}, -1),
    ({'customer_id': 2, 'orders': 1, 'total_price': 50, 'window_end': 175}, -1),
    ({'customer_id': 1, 'orders': 1, 'total_price': 500, 'window_end': 300}, 1)
])
print("-> OK: Old windows [15, 115) and [75, 175) retracted correctly; wew window [200, 300) triggered correctly.")


print("\n=== Step 6: And yet another order from customer 1 arrives too late ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=999, ts=5, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")

print("\n🎉 Done.")

=== Step 1: An order from customer 1 arrives (price=100, ts=10) ===
-> OK. Window [10, 110) triggered correctly.

=== Step 2: Another order from customer 1 arrives (price=200, ts=30) ===
-> OK: Correction for window [10, 110) triggered correctly.

=== Step 3: An order from customer 2 arrives (price=50, ts=75) ===
-> OK: Window [75, 175) triggered correctly.

=== Step 4: Another order from customer 1 arrives late but still inside [10, 110) (price=50, ts=15) ===
-> OK: Window [10, 110) retracted correctly; window [15, 11ß) triggered correctly.

=== Step 5: Yet another order from customer 1 arrives (not inside [15, 115) (price=500, ts=200) ===
-> OK: Old windows [15, 115) and [75, 175) retracted correctly; wew window [200, 300) triggered correctly.

=== Step 6: And yet another order from customer 1 arrives too late ===
-> OK: Order arrived too late - no action.

🎉 Done.


In [123]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

ts_step_int = 1
gap_int = ts_step_int * 20
max_session_int = ts_step_int * 200
allowed_lateness_int = gap_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_session(lambda r: r["ts"], max_session_int, allowed_lateness_int)
    .distinct()
)
#
sink_tn = order_tn.group_by_agg_session(
    ts_fun=lambda r: r["ts"],
    gap_int=gap_int,
    key_fun=lambda r: r["customer_id"],
    agg_fun=lambda agg_r, r: {"orders": agg_r["orders"] + 1,           
                              "total_price": agg_r["total_price"] + r["price"]},
    agg_initial_any={"orders": 0, "total_price": 0},
    project_fun=lambda key_any, agg_r: {"customer_id": key_any,
                                        "orders": agg_r["orders"],
                                        "total_price": agg_r["total_price"]},
                                        trigger_positive_only_bool=False
).sink(sink_str)
#
built_tn = Tn.build(sink_tn)
built_tn.from_zSet(Tn._to_records)


In [124]:
built_tn.reset()

print("=== Step 1: An order from customer 1 arrives (price=100, ts=10) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=100, ts=10, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Window [10, 30) not yet triggered.")


print("\n=== Step 2: Another order from customer 1 arrives (price=200, ts=25) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=200, ts=25, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Window [25, 45) also not yet triggered but merged with [10, 30) => [10, 45).")


print("\n=== Step 3: An order from customer 2 arrives (price=50, ts=75) ===")
r_w_tuple_list = process(built_tn, customer_id=2, price=50, ts=75, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 45}, 1)
])
print("-> OK: Window [10, 45) triggered.")


print("\n=== Step 4: Another order from customer 1 arrives (price=50, ts=15) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=50, ts=15, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 45}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 45}, 1)
])
print("-> OK: Correction for window [10, 45) triggered.")


print("\n=== Step 5: Yet another order from customer 1 arrives (price=500, ts=200) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=500, ts=200, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 95}, 1)
])
print("-> OK: Window [75, 95) triggered.")


print("\n=== Step 6: And yet another order from customer 1 arrives late but not too late (price=999, ts=1) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=999, ts=1, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 45}, -1),
    ({"customer_id": 1, "orders": 4, "total_price": 1349, "window_end": 45}, 1)
])
print("-> OK: Window [10, 45) retracted; New window [1, 45) triggered.")


print("\n=== Step 7: Yet another order from customer 1 arrives (price=100, ts=300) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=100, ts=300, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 4, "total_price": 1349, "window_end": 45}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 95}, -1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 220}, 1)
])
print("-> OK: Windows [1, 45) and [75, 95) retracted; window [200, 220) triggered.")

print("\n=== Step 8: An order from from customer 2 arrives too late (price=200, ts=2) ===")
r_w_tuple_list = process(built_tn, customer_id=2, price=200, ts=2, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")

print("\n🎉 Done.")

=== Step 1: An order from customer 1 arrives (price=100, ts=10) ===
-> OK: Window [10, 30) not yet triggered.

=== Step 2: Another order from customer 1 arrives (price=200, ts=25) ===
-> OK: Window [25, 45) also not yet triggered but merged with [10, 30) => [10, 45).

=== Step 3: An order from customer 2 arrives (price=50, ts=75) ===
-> OK: Window [10, 45) triggered.

=== Step 4: Another order from customer 1 arrives (price=50, ts=15) ===
-> OK: Correction for window [10, 45) triggered.

=== Step 5: Yet another order from customer 1 arrives (price=500, ts=200) ===
-> OK: Window [75, 95) triggered.

=== Step 6: And yet another order from customer 1 arrives late but not too late (price=999, ts=1) ===
-> OK: Window [10, 45) retracted; New window [1, 45) triggered.

=== Step 7: Yet another order from customer 1 arrives (price=100, ts=300) ===
-> OK: Windows [1, 45) and [75, 95) retracted; window [200, 220) triggered.

=== Step 8: An order from from customer 2 arrives too late (price=200, t

In [121]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

ts_step_int = 1
gap_int = ts_step_int * 20
max_session_int = ts_step_int * 200
allowed_lateness_int = gap_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_session(lambda r: r["ts"], max_session_int, allowed_lateness_int)
    .distinct()
)
#
sink_tn = order_tn.group_by_agg_session(
    ts_fun=lambda r: r["ts"],
    gap_int=gap_int,
    key_fun=lambda r: r["customer_id"], 
    agg_fun=lambda agg_r, r: {"orders": agg_r["orders"] + 1,           
                              "total_price": agg_r["total_price"] + r["price"]},
    agg_initial_any={"orders": 0, "total_price": 0},
    project_fun=lambda key_any, agg_r: {"customer_id": key_any,
                                        "orders": agg_r["orders"],
                                        "total_price": agg_r["total_price"]},
    trigger_fun=lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1] or r_end_ts_tuple[0]["total_price"] > 200, 
    trigger_positive_only_bool=False
).sink(sink_str)
#
built_tn = Tn.build(sink_tn)
built_tn.from_zSet(Tn._to_records)


In [122]:
built_tn.reset()

print("=== Step 1: An order from customer 1 arrives (price=100, ts=10) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=100, ts=10, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Window [10, 30) not yet triggered.")


print("\n=== Step 2: Another order from customer 1 arrives (price=200, ts=25) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=200, ts=25, w=1)
assert_output(r_w_tuple_list, [
    ({'customer_id': 1, 'orders': 2, 'total_price': 300, 'window_end': 45}, 1)
])
print("-> OK: Window [25, 45) - and already triggered since total_price >= 300.")


print("\n=== Step 3: An order from customer 2 arrives (price=50, ts=75) ===")
r_w_tuple_list = process(built_tn, customer_id=2, price=50, ts=75, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Window [10, 45) triggered.")


print("\n=== Step 4: Another order from customer 1 arrives (price=50, ts=15) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=50, ts=15, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 45}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 45}, 1)
])
print("-> OK: Correction for window [10, 45) triggered.")


print("\n=== Step 5: Yet another order from customer 1 arrives (price=500, ts=200) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=500, ts=200, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 95}, 1),
    ({'customer_id': 1, 'orders': 1, 'total_price': 500, 'window_end': 220}, 1)
])
print("-> OK: Windows [75, 95) and [200, 220) triggered (the latter has total price >=300).")


print("\n=== Step 6: And yet another order from customer 1 arrives late but not too late (price=999, ts=1) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=999, ts=1, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 45}, -1),
    ({"customer_id": 1, "orders": 4, "total_price": 1349, "window_end": 45}, 1)
])
print("-> OK: Window [10, 45) retracted; New window [1, 45) triggered.")


print("\n=== Step 7: Yet another order from customer 1 arrives (price=100, ts=300) ===")
r_w_tuple_list = process(built_tn, customer_id=1, price=100, ts=300, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 4, "total_price": 1349, "window_end": 45}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 95}, -1),
])
print("-> OK: Windows [1, 45) and [75, 95) retracted; window [200, 220) triggered.")

print("\n=== Step 8: An order from from customer 2 arrives too late (price=200, ts=2) ===")
r_w_tuple_list = process(built_tn, customer_id=2, price=200, ts=2, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")

print("\n🎉 Done.")

=== Step 1: An order from customer 1 arrives (price=100, ts=10) ===
-> OK: Window [10, 30) not yet triggered.

=== Step 2: Another order from customer 1 arrives (price=200, ts=25) ===
-> OK: Window [25, 45) - and already triggered since total_price >= 300.

=== Step 3: An order from customer 2 arrives (price=50, ts=75) ===
-> OK: Window [10, 45) triggered.

=== Step 4: Another order from customer 1 arrives (price=50, ts=15) ===
-> OK: Correction for window [10, 45) triggered.

=== Step 5: Yet another order from customer 1 arrives (price=500, ts=200) ===
-> OK: Windows [75, 95) and [200, 220) triggered (the latter has total price >=300).

=== Step 6: And yet another order from customer 1 arrives late but not too late (price=999, ts=1) ===
-> OK: Window [10, 45) retracted; New window [1, 45) triggered.

=== Step 7: Yet another order from customer 1 arrives (price=100, ts=300) ===
-> OK: Windows [1, 45) and [75, 95) retracted; window [200, 220) triggered.

=== Step 8: An order from from c